# Airbnb Analysis Project Pipeline

This notebook loads the raw Airbnb dataset into SQLite, runs the SQL cleaning pipeline, validates the cleaned data, and exports the final dataset for Tableau.

# Setting up 02_pipeline

In [14]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path

In [15]:
#set project paths

project_root = Path.cwd().parent

raw_data_path = (project_root
                / "data"
                / "raw"
                / "Airbnb_Open_Data.csv"
)

database_path = (project_root
                / "database"
                /"airbnb.db"
)

clean_path = (project_root
              / "sql"
              /"01_clean.ipynb"
)

analysis_path = (project_root
                 /"sql"
                 /"02_analysis.ipynb"
)

tableau_path = (project_root
                 /"data"
                 /"tableau"
                 /"listing_clean.csv"
)

# Load the raw CSV

In [16]:
df_raw = pd.read_csv(raw_data_path)

print('Raw Airbnb data loaded successfully!')
print(f'Rows: {df_raw.shape[0]:,}')
print(f'Columns: {df_raw.shape[1]}')

Raw Airbnb data loaded successfully!
Rows: 102,599
Columns: 26


C:\Users\jayvy\AppData\Local\Temp\ipykernel_10392\2822456596.py:1: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv(raw_data_path)


# Standize column names before SQLite

Since clean SQL column names make the database much easier to deal with, I will just be completing this small preparation in Python

In [17]:
df_raw.columns = (
    df_raw.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

df_raw = df_raw.rename(columns={
    'name': 'listing_name',
    'lat': 'latitude',
    'long': 'longitude'
})

print(df_raw.columns.tolist())

['id', 'listing_name', 'host_id', 'host_identity_verified', 'host_name', 'neighbourhood_group', 'neighbourhood', 'latitude', 'longitude', 'country', 'country_code', 'instant_bookable', 'cancellation_policy', 'room_type', 'construction_year', 'price', 'service_fee', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'review_rate_number', 'calculated_host_listings_count', 'availability_365', 'house_rules', 'license']


# Load Raw Data into SQLite

In [18]:
conn = sqlite3.connect(database_path)

df_raw.to_sql(
    'airbnb_raw',
    conn,
    if_exists='replace',
    index=False
)

print('Raw Airbnb loaded into SQLite successfully!')

Raw Airbnb loaded into SQLite successfully!


# Preview raw data

In [19]:
sql_preview = pd.read_sql_query(
'''
    SELECT *
    FROM airbnb_raw
    LIMIT 5;
''',
    conn
)

display(sql_preview)

,id,listing_name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,latitude,longitude,country,...,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules,license
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,...,$193,10.0,9.0,10/19/2021,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...,None
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,...,$28,30.0,45.0,5/21/2022,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...,None
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,None,Elise,Manhattan,Harlem,40.80902,-73.94190,United States,...,$124,3.0,0.0,None,NaN,5.0,1.0,352.0,"I encourage you to use my kitchen, cooking and...",None
3,1002755,None,85098326012,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,...,$74,30.0,270.0,7/5/2019,4.64,4.0,1.0,322.0,None,None
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,verified,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,United States,...,$41,10.0,9.0,11/19/2018,0.10,3.0,1.0,289.0,"Please no smoking in the house, porch or on th...",None
